# BASE-002 approved LR/MLP convergence rerun

This notebook runs only the approved Gate 3 convergence amendment: LR `max_iter=1000` and MLP `max_iter=200`. It uses CPU, the frozen seed-2027 sample, and every approved calibration/test row. It performs no conformal calibration, statistical test, Flow experiment, or parameter tuning.

Attach the generated `kaggle_rerun_bundle` as a Kaggle Dataset, select **Accelerator: None**, enable Internet for the exact package install, and choose **Save Version → Save & Run All**. Completed chunks resume within the same Kaggle working directory.

In [ ]:
import json
import os
import platform
import subprocess
import sys
from pathlib import Path

override = os.environ.get("BASE002_BUNDLE_ROOT")
markers = (
    [Path(override) / "kaggle_bundle_manifest.json"]
    if override
    else list(Path("/kaggle/input").rglob("kaggle_bundle_manifest.json"))
)
if len(markers) != 1:
    raise RuntimeError(f"Expected exactly one BASE-002 bundle, found {len(markers)}: {markers}")
bundle_root = markers[0].parent
amendment = json.loads((bundle_root / "config/base002_kaggle_rerun.json").read_text())
python_version = '.'.join(platform.python_version_tuple()[:2])
if python_version != amendment["required_python_major_minor"]:
    required = amendment["required_python_major_minor"]
    raise RuntimeError(
        f"This rerun requires Python {required}; current runtime is {python_version}"
    )
requirements = bundle_root / "config/requirements-kaggle-base002.txt"
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", "--requirement", str(requirements)]
)
print(f"Validated bundle location: {bundle_root}")


In [ ]:
output_root = Path(os.environ.get("BASE002_OUTPUT_ROOT", "/kaggle/working/base002_kaggle_rerun"))
environment = os.environ.copy()
environment["PYTHONPATH"] = (
    str(bundle_root / "src")
    + os.pathsep
    + environment.get("PYTHONPATH", "")
)
command = [
    sys.executable,
    "-m",
    "iot_diad.modeling.kaggle_rerun",
    "--bundle-root",
    str(bundle_root),
    "--output-root",
    str(output_root),
]
print("Running:", ' '.join(command))
subprocess.check_call(command, env=environment)


In [ ]:
summary = json.loads((output_root / "rerun_summary.json").read_text())
verification = json.loads((output_root / "rerun_verification.json").read_text())
print(json.dumps(summary, indent=2, sort_keys=True))
print(json.dumps(verification, indent=2, sort_keys=True))
if summary["acceptance_status"] != "passed":
    raise RuntimeError(
        "A convergence warning remains. Preserve outputs and return them unchanged."
    )
print(f"Verified output directory: {output_root}")
print(
    "Save this notebook version with all output files, then retain the complete "
    "output directory for independent verification."
)
